# 变量组方案 × 固定效应组合比较

这个 notebook 用于系统比较：

- 4 套解释变量组
- 3 种固定效应组合
  - `Province = No, Year = Yes`
  - `Province = Yes, Year = No`
  - `Province = Yes, Year = Yes`

统一口径：

- 因变量：`AMR_AGG_z`
- 自变量：按省份中位数 + 全局中位数补缺，再做 z-score 进入回归
- 标准误：按省份聚类稳健 SE
- 目标：在**保留 `R1xday` 与 `抗菌药物使用强度`**的前提下，比较不同变量组和不同固定方式的表现

本 notebook 会输出：

- 长表形式的模型汇总表
- 横向对比总表
- 系数明细表
- 原始尺度 / 标准化后 VIF 对比表
- 每个模型的 Lancet 风格结果表

## R-squared、R-squared (Overall)、R² (within) 的关系

- `R-squared`
  - `PanelOLS` 主结果里报告的模型拟合优度，是当前估计模型对应的“主 R²”。
- `R-squared (Overall)`
  - 忽略固定效应时，因变量与解释变量整体拟合程度的参考指标。
  - 更接近“省际差异 + 省内变化一起看”的整体解释度。
- `R² (within)`
  - 关注在固定效应处理之后，模型对“组内变化”解释得怎么样。
  - 在固定效应模型里通常最严格，也最容易偏低甚至为负。

这三个指标不是互相替代关系，而是从不同角度看同一个模型：

- `R-squared`：主结果摘要
- `R-squared (Overall)`：整体层面参考
- `R² (within)`：固定效应识别层面的核心参考

因此后面的 Lancet 表会把这 3 个指标全部列出来。

In [1]:
import os
import re
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from linearmodels.panel import PanelOLS
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

AMR_PATH = r"C:\Users\lunch\Downloads\amr_rate.csv"
X_PATH = r"C:\Users\lunch\Downloads\climate_social_eco.csv"

AMR_COLS = [
    "MRCNS", "VREFS", "VREFM", "PRSP", "ERSP", "3GCRKP",
    "MRSA", "3GCREC", "CREC", "QREC", "CRPA", "CRKP", "CRAB"
]

FE_SPECS = {
    "NoYes_YearFE": {
        "entity_effects": False,
        "time_effects": True,
        "province_fe": "No",
        "year_fe": "Yes",
        "label": "Province: No / Year: Yes",
    },
    "YesNo_EntityFE": {
        "entity_effects": True,
        "time_effects": False,
        "province_fe": "Yes",
        "year_fe": "No",
        "label": "Province: Yes / Year: No",
    },
    "YesYes_TwoWayFE": {
        "entity_effects": True,
        "time_effects": True,
        "province_fe": "Yes",
        "year_fe": "Yes",
        "label": "Province: Yes / Year: Yes",
    },
}

def to_float(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").astype(float)

def zscore_series(s: pd.Series) -> pd.Series:
    s = to_float(s)
    mu = np.nanmean(s.values)
    sd = np.nanstd(s.values, ddof=0)
    if not np.isfinite(sd) or sd == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sd

def fill_panel_median(df: pd.DataFrame, col: str) -> pd.Series:
    out = to_float(df[col])
    out = out.groupby(df["Province"]).transform(lambda s: s.fillna(s.median()))
    out = out.fillna(out.median())
    return out

def compute_vif(x: pd.DataFrame) -> pd.DataFrame:
    vals = x.values.astype(float)
    return pd.DataFrame({
        "feature": x.columns,
        "VIF": [variance_inflation_factor(vals, i) for i in range(len(x.columns))]
    })

def p_to_text(p: float) -> str:
    if not np.isfinite(p):
        return ""
    if p < 1e-4:
        return "<0.0001"
    return f"{p:.4f}".rstrip("0").rstrip(".")

def p_to_stars(p: float) -> str:
    if not np.isfinite(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

def fmt_coef(x: float, digits=4, stars="") -> str:
    if not np.isfinite(x):
        return ""
    return f"{x:.{digits}f}".rstrip("0").rstrip(".") + stars

def fmt_ci(lo: float, hi: float, digits=4) -> str:
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return ""
    return f"({lo:.{digits}f}, {hi:.{digits}f})"

def safe_sheet_name(name: str) -> str:
    name = re.sub(r"[\\/*?:\[\]]", "_", name)
    return name[:31]

def build_lancet_table(res, x_cols, scheme_name: str, fe_name: str, fe_cfg: dict) -> pd.DataFrame:
    ci = res.conf_int()
    ci_lo = ci.iloc[:, 0]
    ci_hi = ci.iloc[:, 1]

    rows = []
    for xname in x_cols:
        b = float(res.params.get(xname, np.nan))
        p = float(res.pvalues.get(xname, np.nan))
        lo = float(ci_lo.loc[xname]) if xname in ci_lo.index else np.nan
        hi = float(ci_hi.loc[xname]) if xname in ci_hi.index else np.nan
        rows.append({
            "Predictor": xname,
            "Coefficient": fmt_coef(b, 4, p_to_stars(p)),
            "95% CI": fmt_ci(lo, hi, 4),
            "p value": p_to_text(p),
        })

    rows.extend([
        {"Predictor": "Province", "Coefficient": fe_cfg["province_fe"], "95% CI": "", "p value": ""},
        {"Predictor": "Year", "Coefficient": fe_cfg["year_fe"], "95% CI": "", "p value": ""},
        {"Predictor": "R-squared", "Coefficient": fmt_coef(float(res.rsquared), 3), "95% CI": "", "p value": ""},
        {"Predictor": "R-squared (Overall)", "Coefficient": fmt_coef(float(res.rsquared_overall), 3), "95% CI": "", "p value": ""},
        {"Predictor": "R² (within)", "Coefficient": fmt_coef(float(res.rsquared_within), 3), "95% CI": "", "p value": ""},
        {"Predictor": "Total number of observations", "Coefficient": str(int(res.nobs)), "95% CI": "", "p value": ""},
    ])
    return pd.DataFrame(rows)

def build_lancet_export_block(table: pd.DataFrame, scheme_name: str, fe_cfg: dict) -> pd.DataFrame:
    title_row = pd.DataFrame([{
        "Predictor": f"{scheme_name} | {fe_cfg['label']}",
        "Coefficient": "",
        "95% CI": "",
        "p value": "",
    }])
    spacer_row = pd.DataFrame([{
        "Predictor": "",
        "Coefficient": "",
        "95% CI": "",
        "p value": "",
    }])
    return pd.concat([title_row, table, spacer_row], ignore_index=True)

def explain_result(row: pd.Series) -> str:
    sig_txt = row["sig_predictors_p_lt_0_05"] if row["sig_predictors_p_lt_0_05"] else "无变量达到 p<0.05"
    return (
        f"**结果解释**  \n"
        f"- 固定效应设定：{row['fe_label']}  \n"
        f"- 综合排序：第 {int(row['performance_rank'])} 名，综合得分={row['performance_score']:.3f}  \n"
        f"- 主模型 `R-squared={row['r2_model']:.3f}`，`R-squared (Overall)={row['r2_overall']:.3f}`，`R² (within)={row['r2_within']:.3f}`  \n"
        f"- `R1xday` 系数={row['coef_R1xday']:.4f}，p={row['p_R1xday']:.4f}  \n"
        f"- `抗菌药物使用强度` 系数={row['coef_AMC']:.4f}，p={row['p_AMC']:.4f}  \n"
        f"- 显著变量：{sig_txt}  \n"
        f"- VIF：raw 最大值={row['max_vif_raw']:.2f}，z-score 后最大值={row['max_vif_z']:.2f}"
    )


In [2]:
amr = pd.read_csv(AMR_PATH, encoding="utf-8-sig")
x = pd.read_csv(X_PATH, encoding="utf-8-sig")

# 显式固定 climate 表前两列为 Province / Year，避免中英文列名混杂
x = x.rename(columns={x.columns[0]: "Province", x.columns[1]: "Year"})

for df_temp in [amr, x]:
    if "Province" not in df_temp.columns:
        raise ValueError("Province column not found")
    if "Year" not in df_temp.columns:
        raise ValueError("Year column not found")
    df_temp["Province"] = df_temp["Province"].astype(str).str.strip()
    df_temp["Year"] = pd.to_numeric(df_temp["Year"], errors="coerce").astype("Int64")

df = amr.merge(x, on=["Province", "Year"], how="inner")
df = df[df["Year"].between(2014, 2023)].dropna(subset=["Province", "Year"]).copy()

for c in AMR_COLS:
    df[c] = to_float(df[c])

df["AMR_AGG"] = df[AMR_COLS].mean(axis=1, skipna=True)
z_amr = pd.DataFrame({c: zscore_series(df[c]) for c in AMR_COLS})
df["AMR_AGG_z"] = z_amr.mean(axis=1, skipna=True)

print("Merged shape:", df.shape)
print("Years:", int(df["Year"].min()), "-", int(df["Year"].max()))
print("Provinces:", df["Province"].nunique())

Merged shape: (310, 46)
Years: 2014 - 2023
Provinces: 31


## 候选变量组

这里先比较 4 套候选方案：

- `方案A_平衡主线组`
- `方案D_城市气候组`
- `方案F_低VIF主线组`
- `方案C_污染替代组`

每套方案都会继续与 3 种固定效应组合交叉，形成 12 个模型。

In [3]:
SCHEMES = {
    "方案A_平衡主线组": {
        "note": "推荐主线候选：R1xday 与 AMC 都较稳，VIF 仍在可接受范围内。",
        "vars": ["省平均气温", "R1xday", "PM2.5", "抗菌药物使用强度", "GDP", "文盲比例", "人均日生活用水量(升)", "牲畜饲养\n-猪年底头数"],
    },
    "方案D_城市气候组": {
        "note": "用主要城市平均气温和绿化覆盖率替换部分省级代理，适合做城市环境视角的敏感性分析。",
        "vars": ["主要城市平均气温", "R1xday", "PM2.5", "抗菌药物使用强度", "GDP", "文盲比例", "建成区绿化覆盖率", "牲畜饲养\n-猪年底头数"],
    },
    "方案F_低VIF主线组": {
        "note": "当前 VIF 最低的一组，适合做低共线性参考版本。",
        "vars": ["TA（°C）", "R1xday", "PM2.5", "抗菌药物使用强度", "GDP", "文盲比例", "人均日生活用水量(升)", "牲畜饲养\n-猪年底头数", "主要城市日照时数"],
    },
    "方案C_污染替代组": {
        "note": "用氮氧化物和可支配收入替换 PM2.5 / GDP 的一种替代规格。",
        "vars": ["省平均气温", "R1xday", "氮氧化物", "抗菌药物使用强度", "可支配收入", "建成区绿化覆盖率", "人均日生活用水量(升)", "牲畜饲养\n-猪年底头数"],
    },
}

In [4]:
def run_model(df_raw: pd.DataFrame, scheme_name: str, scheme_spec: dict, fe_name: str, fe_cfg: dict):
    cols = scheme_spec["vars"]
    work = df_raw.copy()
    raw_x = pd.DataFrame(index=work.index)

    for c in cols:
        raw_x[c] = fill_panel_median(work, c)
        work[c] = zscore_series(raw_x[c])

    z_x = work[cols].copy()
    vif_raw = compute_vif(raw_x).rename(columns={"VIF": "vif_raw"})
    vif_z = compute_vif(z_x).rename(columns={"VIF": "vif_z"})
    vif_table = vif_raw.merge(vif_z, on="feature", how="inner")
    vif_table["abs_diff"] = (vif_table["vif_raw"] - vif_table["vif_z"]).abs()

    panel = work.set_index(["Province", "Year"]).sort_index()
    tmp = pd.concat([panel["AMR_AGG_z"], panel[cols]], axis=1).dropna()
    y = tmp["AMR_AGG_z"]
    x_model = tmp[cols]

    res = PanelOLS(
        y,
        x_model,
        entity_effects=fe_cfg["entity_effects"],
        time_effects=fe_cfg["time_effects"],
    ).fit(cov_type="clustered", cluster_entity=True)

    summary_row = {
        "scheme": scheme_name,
        "scheme_note": scheme_spec["note"],
        "fe_spec": fe_name,
        "fe_label": fe_cfg["label"],
        "province_fe": fe_cfg["province_fe"],
        "year_fe": fe_cfg["year_fe"],
        "n_vars": len(cols),
        "variables": " | ".join(cols),
        "nobs": int(res.nobs),
        "r2_model": float(res.rsquared),
        "r2_overall": float(res.rsquared_overall),
        "r2_within": float(res.rsquared_within),
        "max_vif_raw": float(vif_table["vif_raw"].max()),
        "median_vif_raw": float(vif_table["vif_raw"].median()),
        "max_vif_z": float(vif_table["vif_z"].max()),
        "median_vif_z": float(vif_table["vif_z"].median()),
        "max_abs_vif_diff": float(vif_table["abs_diff"].max()),
        "coef_R1xday": float(res.params["R1xday"]),
        "p_R1xday": float(res.pvalues["R1xday"]),
        "coef_AMC": float(res.params["抗菌药物使用强度"]),
        "p_AMC": float(res.pvalues["抗菌药物使用强度"]),
        "sig_predictors_p_lt_0_05": ", ".join([c for c in cols if float(res.pvalues[c]) < 0.05]),
    }

    coef_table = pd.DataFrame({
        "scheme": scheme_name,
        "fe_spec": fe_name,
        "predictor": res.params.index,
        "coef": res.params.values,
        "p_value": res.pvalues.values,
        "ci_low": res.conf_int().iloc[:, 0].values,
        "ci_high": res.conf_int().iloc[:, 1].values,
    })

    vif_table = vif_table.rename(columns={"feature": "predictor"})
    vif_table["scheme"] = scheme_name
    vif_table["fe_spec"] = fe_name

    tab_lancet = build_lancet_table(res, cols, scheme_name, fe_name, fe_cfg)
    return summary_row, coef_table, vif_table, tab_lancet


summary_rows = []
coef_tables = []
vif_tables = []
lancet_tables = {}
lancet_export_blocks = []

for scheme_name, scheme_spec in SCHEMES.items():
    for fe_name, fe_cfg in FE_SPECS.items():
        summary_row, coef_table, vif_table, tab_lancet = run_model(df, scheme_name, scheme_spec, fe_name, fe_cfg)
        summary_rows.append(summary_row)
        coef_tables.append(coef_table)
        vif_tables.append(vif_table)
        lancet_tables[(scheme_name, fe_name)] = tab_lancet
        lancet_export_blocks.append(build_lancet_export_block(tab_lancet, scheme_name, fe_cfg))

summary_df = pd.DataFrame(summary_rows).sort_values(["scheme", "fe_spec"])
coef_df = pd.concat(coef_tables, ignore_index=True)
vif_df = pd.concat(vif_tables, ignore_index=True)
lancet_long_df = pd.concat(lancet_export_blocks, ignore_index=True)

summary_df

,scheme,scheme_note,fe_spec,fe_label,province_fe,year_fe,n_vars,variables,nobs,r2_model,r2_overall,r2_within,max_vif_raw,median_vif_raw,max_vif_z,median_vif_z,max_abs_vif_diff,coef_R1xday,p_R1xday,coef_AMC,p_AMC,sig_predictors_p_lt_0_05
0,方案A_平衡主线组,推荐主线候选：R1xday 与 AMC 都较稳，VIF 仍在可接受范围内。,NoYes_YearFE,Province: No / Year: Yes,No,Yes,8,省平均气温 | R1xday | PM2.5 | 抗菌药物使用强度 | GDP | 文盲比例...,307,0.333815,0.274503,-0.185709,24.796987,4.779977,3.054480,1.670002,22.644996,0.150825,0.042610,0.131034,0.003385,"R1xday, 抗菌药物使用强度, GDP"
1,方案A_平衡主线组,推荐主线候选：R1xday 与 AMC 都较稳，VIF 仍在可接受范围内。,YesNo_EntityFE,Province: Yes / Year: No,Yes,No,8,省平均气温 | R1xday | PM2.5 | 抗菌药物使用强度 | GDP | 文盲比例...,307,0.454837,-2.537903,0.454837,24.796987,4.779977,3.054480,1.670002,22.644996,-0.020615,0.300460,0.002389,0.899995,"PM2.5, GDP, 文盲比例, 牲畜饲养\n-猪年底头数"
2,方案A_平衡主线组,推荐主线候选：R1xday 与 AMC 都较稳，VIF 仍在可接受范围内。,YesYes_TwoWayFE,Province: Yes / Year: Yes,Yes,Yes,8,省平均气温 | R1xday | PM2.5 | 抗菌药物使用强度 | GDP | 文盲比例...,307,0.021212,-0.418011,0.025958,24.796987,4.779977,3.054480,1.670002,22.644996,-0.003308,0.850688,0.009285,0.516960,
9,方案C_污染替代组,用氮氧化物和可支配收入替换 PM2.5 / GDP 的一种替代规格。,NoYes_YearFE,Province: No / Year: Yes,No,Yes,8,省平均气温 | R1xday | 氮氧化物 | 抗菌药物使用强度 | 可支配收入 | 建成区...,307,0.531999,0.269155,-0.660241,62.516300,9.843330,2.332034,1.520134,60.990177,0.048007,0.330806,0.111917,0.002199,"抗菌药物使用强度, 可支配收入, 建成区绿化覆盖率"
10,方案C_污染替代组,用氮氧化物和可支配收入替换 PM2.5 / GDP 的一种替代规格。,YesNo_EntityFE,Province: Yes / Year: No,Yes,No,8,省平均气温 | R1xday | 氮氧化物 | 抗菌药物使用强度 | 可支配收入 | 建成区...,307,0.606516,-1.070298,0.606516,62.516300,9.843330,2.332034,1.520134,60.990177,0.008516,0.508686,-0.012532,0.443208,"氮氧化物, 可支配收入"
11,方案C_污染替代组,用氮氧化物和可支配收入替换 PM2.5 / GDP 的一种替代规格。,YesYes_TwoWayFE,Province: Yes / Year: Yes,Yes,Yes,8,省平均气温 | R1xday | 氮氧化物 | 抗菌药物使用强度 | 可支配收入 | 建成区...,307,0.042709,-0.520339,-0.376369,62.516300,9.843330,2.332034,1.520134,60.990177,-0.003663,0.833159,0.008885,0.523672,
3,方案D_城市气候组,用主要城市平均气温和绿化覆盖率替换部分省级代理，适合做城市环境视角的敏感性分析。,NoYes_YearFE,Province: No / Year: Yes,No,Yes,8,主要城市平均气温 | R1xday | PM2.5 | 抗菌药物使用强度 | GDP | 文...,307,0.408236,0.191679,-0.651929,47.238880,4.511373,2.088800,1.460562,45.773200,0.151939,0.063183,0.119722,0.001181,"主要城市平均气温, 抗菌药物使用强度, 建成区绿化覆盖率"
4,方案D_城市气候组,用主要城市平均气温和绿化覆盖率替换部分省级代理，适合做城市环境视角的敏感性分析。,YesNo_EntityFE,Province: Yes / Year: No,Yes,No,8,主要城市平均气温 | R1xday | PM2.5 | 抗菌药物使用强度 | GDP | 文...,307,0.481033,-0.828004,0.481033,47.238880,4.511373,2.088800,1.460562,45.773200,-0.012699,0.590733,-0.009800,0.599654,"PM2.5, GDP, 文盲比例, 建成区绿化覆盖率"
5,方案D_城市气候组,用主要城市平均气温和绿化覆盖率替换部分省级代理，适合做城市环境视角的敏感性分析。,YesYes_TwoWayFE,Province: Yes / Year: Yes,Yes,Yes,8,主要城市平均气温 | R1xday | PM2.5 | 抗菌药物使用强度 | GDP | 文...,307,0.038507,-0.096491,-0.138485,47.238880,4.511373,2.088800,1.460562,45.773200,-0.009058,0.604125,0.009943,0.494008,文盲比例
6,方案F_低VIF主线组,当前 VIF 最低的一组，适合做低共线性参考版本。,NoYes_YearFE,Province: No / Year: Yes,No,Yes,9,TA（°C） | R1xday | PM2.5 | 抗菌药物使用强度 | GDP | 文盲比...,307,0.345062,0.228494,-0.366308,32.637482,4.575283,1.820431,1.572431,31.517528,0.134372,0.051118,0.128255,0.003374,"抗菌药物使用强度, GDP"


## 横向对比总表

下面这张表把 12 个模型的关键指标横向展开，适合快速比较：

- `R-squared`
- `R-squared (Overall)`
- `R² (within)`
- `R1xday` 的系数和 p 值
- `抗菌药物使用强度` 的系数和 p 值
- 最大 VIF（raw / z-score 后）

In [5]:
horizontal_metrics = [
    "r2_model",
    "r2_overall",
    "r2_within",
    "coef_R1xday",
    "p_R1xday",
    "coef_AMC",
    "p_AMC",
    "max_vif_raw",
    "max_vif_z",
]

summary_df["model_id"] = summary_df["scheme"] + " | " + summary_df["fe_label"]
horizontal_compare_df = (
    summary_df[["model_id"] + horizontal_metrics]
    .set_index("model_id")
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)

horizontal_compare_df

model_id,metric,方案A_平衡主线组 | Province: No / Year: Yes,方案A_平衡主线组 | Province: Yes / Year: No,方案A_平衡主线组 | Province: Yes / Year: Yes,方案C_污染替代组 | Province: No / Year: Yes,方案C_污染替代组 | Province: Yes / Year: No,方案C_污染替代组 | Province: Yes / Year: Yes,方案D_城市气候组 | Province: No / Year: Yes,方案D_城市气候组 | Province: Yes / Year: No,方案D_城市气候组 | Province: Yes / Year: Yes,方案F_低VIF主线组 | Province: No / Year: Yes,方案F_低VIF主线组 | Province: Yes / Year: No,方案F_低VIF主线组 | Province: Yes / Year: Yes
0,r2_model,0.333815,0.454837,0.021212,0.531999,0.606516,0.042709,0.408236,0.481033,0.038507,0.345062,0.454924,0.021466
1,r2_overall,0.274503,-2.537903,-0.418011,0.269155,-1.070298,-0.520339,0.191679,-0.828004,-0.096491,0.228494,-0.738865,-0.205236
2,r2_within,-0.185709,0.454837,0.025958,-0.660241,0.606516,-0.376369,-0.651929,0.481033,-0.138485,-0.366308,0.454924,0.023311
3,coef_R1xday,0.150825,-0.020615,-0.003308,0.048007,0.008516,-0.003663,0.151939,-0.012699,-0.009058,0.134372,-0.020747,-0.003453
4,p_R1xday,0.042610,0.300460,0.850688,0.330806,0.508686,0.833159,0.063183,0.590733,0.604125,0.051118,0.302745,0.845380
5,coef_AMC,0.131034,0.002389,0.009285,0.111917,-0.012532,0.008885,0.119722,-0.009800,0.009943,0.128255,0.002518,0.009281
6,p_AMC,0.003385,0.899995,0.516960,0.002199,0.443208,0.523672,0.001181,0.599654,0.494008,0.003374,0.895193,0.517364
7,max_vif_raw,24.796987,24.796987,24.796987,62.516300,62.516300,62.516300,47.238880,47.238880,47.238880,32.637482,32.637482,32.637482
8,max_vif_z,3.054480,3.054480,3.054480,2.332034,2.332034,2.332034,2.088800,2.088800,2.088800,1.820431,1.820431,1.820431


## 模型性能排序

这里额外给出一个综合排序，方便快速挑主模型。这个排序不是单看某一个 `R²`，而是把三类信息一起考虑：

- `45%`：核心变量支持度
  - 优先奖励 `R1xday` 与 `抗菌药物使用强度` 系数为正、且 `p value` 更小的模型
- `35%`：模型拟合度
  - 综合 `R-squared`、`R-squared (Overall)`、`R² (within)`
- `20%`：共线性
  - `max_vif_z` 越低越好

这是一张“便于筛选”的工作排序表，不替代你对主文模型和附录模型的最终判断。

In [6]:
def minmax01(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    if np.isclose(s.max(), s.min()):
        return pd.Series(np.ones(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

summary_df["r1xday_support_raw"] = np.sign(summary_df["coef_R1xday"]) * (
    -np.log10(summary_df["p_R1xday"].clip(lower=1e-12))
)
summary_df["amc_support_raw"] = np.sign(summary_df["coef_AMC"]) * (
    -np.log10(summary_df["p_AMC"].clip(lower=1e-12))
)
summary_df["core_positive_count"] = (
    (summary_df["coef_R1xday"] > 0).astype(int) +
    (summary_df["coef_AMC"] > 0).astype(int)
)
summary_df["core_sig_count_p_lt_0_05"] = (
    (summary_df["p_R1xday"] < 0.05).astype(int) +
    (summary_df["p_AMC"] < 0.05).astype(int)
)
summary_df["core_signal_score"] = (
    minmax01(summary_df["r1xday_support_raw"]) +
    minmax01(summary_df["amc_support_raw"])
) / 2
summary_df["fit_score"] = (
    minmax01(summary_df["r2_model"]) +
    minmax01(summary_df["r2_overall"]) +
    minmax01(summary_df["r2_within"])
) / 3
summary_df["vif_score"] = 1 - minmax01(np.log1p(summary_df["max_vif_z"]))
summary_df["performance_score"] = (
    0.45 * summary_df["core_signal_score"] +
    0.35 * summary_df["fit_score"] +
    0.20 * summary_df["vif_score"]
)
summary_df["performance_rank"] = (
    summary_df["performance_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

ranking_df = (
    summary_df[[
        "performance_rank",
        "model_id",
        "scheme",
        "fe_label",
        "performance_score",
        "core_signal_score",
        "fit_score",
        "vif_score",
        "core_positive_count",
        "core_sig_count_p_lt_0_05",
        "coef_R1xday",
        "p_R1xday",
        "coef_AMC",
        "p_AMC",
        "r2_model",
        "r2_overall",
        "r2_within",
        "max_vif_z",
        "sig_predictors_p_lt_0_05",
    ]]
    .sort_values(["performance_rank", "performance_score"], ascending=[True, False])
    .reset_index(drop=True)
)

ranking_df

,performance_rank,model_id,scheme,fe_label,performance_score,core_signal_score,fit_score,vif_score,core_positive_count,core_sig_count_p_lt_0_05,coef_R1xday,p_R1xday,coef_AMC,p_AMC,r2_model,r2_overall,r2_within,max_vif_z,sig_predictors_p_lt_0_05
0,1,方案F_低VIF主线组 | Province: No / Year: Yes,方案F_低VIF主线组,Province: No / Year: Yes,0.815711,0.909621,0.589660,1.000000,2,1,0.134372,0.051118,0.128255,0.003374,0.345062,0.228494,-0.366308,1.820431,"抗菌药物使用强度, GDP"
1,2,方案D_城市气候组 | Province: No / Year: Yes,方案D_城市气候组,Province: No / Year: Yes,0.770714,0.954803,0.546116,0.749560,2,1,0.151939,0.063183,0.119722,0.001181,0.408236,0.191679,-0.651929,2.088800,"主要城市平均气温, 抗菌药物使用强度, 建成区绿化覆盖率"
2,3,方案C_污染替代组 | Province: No / Year: Yes,方案C_污染替代组,Province: No / Year: Yes,0.652066,0.723703,0.623595,0.540705,2,1,0.048007,0.330806,0.111917,0.002199,0.531999,0.269155,-0.660241,2.332034,"抗菌药物使用强度, 可支配收入, 建成区绿化覆盖率"
3,4,方案A_平衡主线组 | Province: No / Year: Yes,方案A_平衡主线组,Province: No / Year: Yes,0.641309,0.930285,0.636230,0.000000,2,2,0.150825,0.042610,0.131034,0.003385,0.333815,0.274503,-0.185709,3.054480,"R1xday, 抗菌药物使用强度, GDP"
4,5,方案C_污染替代组 | Province: Yes / Year: No,方案C_污染替代组,Province: Yes / Year: No,0.499331,0.215502,0.840611,0.540705,1,0,0.008516,0.508686,-0.012532,0.443208,0.606516,-1.070298,0.606516,2.332034,"氮氧化物, 可支配收入"
5,6,方案F_低VIF主线组 | Province: Yes / Year: No,方案F_低VIF主线组,Province: Yes / Year: No,0.491705,0.062046,0.753671,1.000000,1,0,-0.020747,0.302745,0.002518,0.895193,0.454924,-0.738865,0.454924,1.820431,"PM2.5, GDP, 文盲比例, 牲畜饲养\n-猪年底头数"
6,7,方案D_城市气候组 | Province: Yes / Year: No,方案D_城市气候组,Province: Yes / Year: No,0.461514,0.097569,0.764845,0.749560,0,0,-0.012699,0.590733,-0.009800,0.599654,0.481033,-0.828004,0.481033,2.088800,"PM2.5, GDP, 文盲比例, 建成区绿化覆盖率"
7,8,方案F_低VIF主线组 | Province: Yes / Year: Yes,方案F_低VIF主线组,Province: Yes / Year: Yes,0.457036,0.216144,0.456488,1.000000,1,0,-0.003453,0.845380,0.009281,0.517364,0.021466,-0.205236,0.023311,1.820431,
8,9,方案D_城市气候组 | Province: Yes / Year: Yes,方案D_城市气候组,Province: Yes / Year: Yes,0.383983,0.180652,0.436506,0.749560,1,0,-0.009058,0.604125,0.009943,0.494008,0.038507,-0.096491,-0.138485,2.088800,文盲比例
9,10,方案C_污染替代组 | Province: Yes / Year: Yes,方案C_污染替代组,Province: Yes / Year: Yes,0.318417,0.213672,0.326067,0.540705,1,0,-0.003663,0.833159,0.008885,0.523672,0.042709,-0.520339,-0.376369,2.332034,


## Lancet 风格结果表

下面会把 12 个模型逐张打印。每张表都包含：

- 表标题中显示 `scheme` 和固定效应组合
- 表内只保留 `Predictor / Coefficient / 95% CI / p value` 4 列
- `Province`
- `Year`
- `R-squared`
- `R-squared (Overall)`
- `R² (within)`
- `Total number of observations`

导出的 `all_tables` 也不再额外保留 `scheme` 或 `fe_spec` 两列，而是用每张表的首行标题来区分模型。

因此你可以直接在 notebook 里纵向看单个模型，也可以通过上面的横向总表横着比较。

In [7]:
for scheme_name, scheme_spec in SCHEMES.items():
    display(Markdown(f"## {scheme_name}"))
    display(Markdown(scheme_spec["note"]))
    for fe_name, fe_cfg in FE_SPECS.items():
        row = summary_df[(summary_df["scheme"] == scheme_name) & (summary_df["fe_spec"] == fe_name)].iloc[0]
        display(Markdown(f"### {fe_cfg['label']}"))
        display(Markdown(explain_result(row)))
        display(lancet_tables[(scheme_name, fe_name)])

## 方案A_平衡主线组

推荐主线候选：R1xday 与 AMC 都较稳，VIF 仍在可接受范围内。

### Province: No / Year: Yes

**结果解释**  
- 固定效应设定：Province: No / Year: Yes  
- 综合排序：第 4 名，综合得分=0.641  
- 主模型 `R-squared=0.334`，`R-squared (Overall)=0.275`，`R² (within)=-0.186`  
- `R1xday` 系数=0.1508，p=0.0426  
- `抗菌药物使用强度` 系数=0.1310，p=0.0034  
- 显著变量：R1xday, 抗菌药物使用强度, GDP  
- VIF：raw 最大值=24.80，z-score 后最大值=3.05

,Predictor,Coefficient,95% CI,p value
0,省平均气温,-0.0336,"(-0.2133, 0.1460)",0.7128
1,R1xday,0.1508*,"(0.0051, 0.2966)",0.0426
2,PM2.5,0.0209,"(-0.1618, 0.2035)",0.8223
3,抗菌药物使用强度,0.131**,"(0.0438, 0.2183)",0.0034
4,GDP,0.1811*,"(0.0250, 0.3371)",0.0231
5,文盲比例,0.007,"(-0.1794, 0.1934)",0.941
6,人均日生活用水量(升),-0.1082,"(-0.2859, 0.0696)",0.2319
7,牲畜饲养\n-猪年底头数,0.0566,"(-0.0926, 0.2059)",0.4556
8,Province,No,,
9,Year,Yes,,


...

**结果解释**  
- 固定效应设定：Province: Yes / Year: No  
- 综合排序：第 5 名，综合得分=0.499  
- 主模型 `R-squared=0.607`，`R-squared (Overall)=-1.070`，`R² (within)=0.607`  
- `R1xday` 系数=0.0085，p=0.5087  
- `抗菌药物使用强度` 系数=-0.0125，p=0.4432  
- 显著变量：氮氧化物, 可支配收入  
- VIF：raw 最大值=62.52，z-score 后最大值=2.33

,Predictor,Coefficient,95% CI,p value
0,省平均气温,0.7008,"(-0.2582, 1.6599)",0.1514
1,R1xday,0.0085,"(-0.0168, 0.0339)",0.5087
2,氮氧化物,0.2344***,"(0.1297, 0.3391)",<0.0001
3,抗菌药物使用强度,-0.0125,"(-0.0447, 0.0196)",0.4432
4,可支配收入,-0.2876***,"(-0.3896, -0.1855)",<0.0001
5,建成区绿化覆盖率,-0.0678,"(-0.1586, 0.0230)",0.1429
6,人均日生活用水量(升),-0.0312,"(-0.1166, 0.0542)",0.4726
7,牲畜饲养\n-猪年底头数,0.1086,"(-0.0176, 0.2348)",0.0914
8,Province,Yes,,
9,Year,No,,


### Province: Yes / Year: Yes

**结果解释**  
- 固定效应设定：Province: Yes / Year: Yes  
- 综合排序：第 10 名，综合得分=0.318  
- 主模型 `R-squared=0.043`，`R-squared (Overall)=-0.520`，`R² (within)=-0.376`  
- `R1xday` 系数=-0.0037，p=0.8332  
- `抗菌药物使用强度` 系数=0.0089，p=0.5237  
- 显著变量：无变量达到 p<0.05  
- VIF：raw 最大值=62.52，z-score 后最大值=2.33

,Predictor,Coefficient,95% CI,p value
0,省平均气温,0.5507,"(-0.8567, 1.9580)",0.4417
1,R1xday,-0.0037,"(-0.0379, 0.0305)",0.8332
2,氮氧化物,0.0412,"(-0.0513, 0.1338)",0.3812
3,抗菌药物使用强度,0.0089,"(-0.0185, 0.0363)",0.5237
4,可支配收入,0.0794,"(-0.1127, 0.2714)",0.4165
5,建成区绿化覆盖率,0.0989,"(-0.0216, 0.2194)",0.1074
6,人均日生活用水量(升),0.0128,"(-0.0861, 0.1117)",0.7991
7,牲畜饲养\n-猪年底头数,-0.0849,"(-0.2735, 0.1037)",0.3763
8,Province,Yes,,
9,Year,Yes,,


In [8]:
summary_csv = os.path.join(RESULT_DIR, "variable_group_scheme_summary.csv")
summary_xlsx = os.path.join(RESULT_DIR, "variable_group_scheme_summary.xlsx")
coef_csv = os.path.join(RESULT_DIR, "variable_group_scheme_coefficients.csv")
coef_xlsx = os.path.join(RESULT_DIR, "variable_group_scheme_coefficients.xlsx")
vif_csv = os.path.join(RESULT_DIR, "variable_group_scheme_vif.csv")
vif_xlsx = os.path.join(RESULT_DIR, "variable_group_scheme_vif.xlsx")
lancet_csv = os.path.join(RESULT_DIR, "variable_group_scheme_lancet_tables.csv")
lancet_xlsx = os.path.join(RESULT_DIR, "variable_group_scheme_lancet_tables.xlsx")
horizontal_csv = os.path.join(RESULT_DIR, "variable_group_scheme_horizontal_compare.csv")
horizontal_xlsx = os.path.join(RESULT_DIR, "variable_group_scheme_horizontal_compare.xlsx")
ranking_csv = os.path.join(RESULT_DIR, "variable_group_scheme_ranking.csv")
ranking_xlsx = os.path.join(RESULT_DIR, "variable_group_scheme_ranking.xlsx")

summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
summary_df.to_excel(summary_xlsx, index=False)
coef_df.to_csv(coef_csv, index=False, encoding="utf-8-sig")
coef_df.to_excel(coef_xlsx, index=False)
vif_df.to_csv(vif_csv, index=False, encoding="utf-8-sig")
vif_df.to_excel(vif_xlsx, index=False)
lancet_long_df.to_csv(lancet_csv, index=False, encoding="utf-8-sig")
horizontal_compare_df.to_csv(horizontal_csv, index=False, encoding="utf-8-sig")
ranking_df.to_csv(ranking_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(lancet_xlsx) as writer:
    lancet_long_df.to_excel(writer, sheet_name="all_tables", index=False)
    for (scheme_name, fe_name), table in lancet_tables.items():
        sheet = safe_sheet_name(f"{scheme_name}_{fe_name}")
        table.to_excel(writer, sheet_name=sheet, index=False)

horizontal_compare_df.to_excel(horizontal_xlsx, index=False)
ranking_df.to_excel(ranking_xlsx, index=False)

print("Saved:", summary_csv)
print("Saved:", summary_xlsx)
print("Saved:", coef_csv)
print("Saved:", coef_xlsx)
print("Saved:", vif_csv)
print("Saved:", vif_xlsx)
print("Saved:", lancet_csv)
print("Saved:", lancet_xlsx)
print("Saved:", horizontal_csv)
print("Saved:", horizontal_xlsx)
print("Saved:", ranking_csv)
print("Saved:", ranking_xlsx)

Saved: results\variable_group_scheme_summary.csv
Saved: results\variable_group_scheme_summary.xlsx
Saved: results\variable_group_scheme_coefficients.csv
Saved: results\variable_group_scheme_coefficients.xlsx
Saved: results\variable_group_scheme_vif.csv
Saved: results\variable_group_scheme_vif.xlsx
Saved: results\variable_group_scheme_lancet_tables.csv
Saved: results\variable_group_scheme_lancet_tables.xlsx
Saved: results\variable_group_scheme_horizontal_compare.csv
Saved: results\variable_group_scheme_horizontal_compare.xlsx
Saved: results\variable_group_scheme_ranking.csv
Saved: results\variable_group_scheme_ranking.xlsx


## 怎么看这些结果

- 先看 `variable_group_scheme_ranking.xlsx`
  - 适合先快速筛出综合表现最好的模型
- 再看 `variable_group_scheme_horizontal_compare.xlsx`
  - 适合一眼比较 12 个模型的核心统计量
- 再看 `variable_group_scheme_summary.xlsx`
  - 适合看长表形式的全量汇总
- 再看 `variable_group_scheme_lancet_tables.xlsx`
  - `all_tables` 用首行标题区分模型；各单独 sheet 则是一张模型一张表
- 如果你要检查共线性，再看 `variable_group_scheme_vif.xlsx`

通常可以按这个顺序筛主模型：

1. `R1xday` 和 `抗菌药物使用强度` 是否方向稳定
2. 这两个变量是否显著或接近显著
3. `R-squared (Overall)` 和 `R² (within)` 是否至少在可接受区间
4. `max_vif_z` 是否仍然偏高
5. 再决定哪套放主文、哪套放附录